In [1]:
# AB TEST 

In [2]:
import numpy as np
import pandas as pd

In [3]:
np.random.seed(42)

In [4]:
N = 80000

In [5]:
df = pd.DataFrame({
    'user_id': range(N),
    'tenure_months': np.random.exponential(24,N).clip(0,120),
    'prior_engagement':np.random.beta(2,23,N),
    'account_type': np.random.choice(['basic', 'premium'], N, p=[0.75,0.25])
})

In [6]:
df.head()

,user_id,tenure_months,prior_engagement,account_type
0,0,11.262434,0.050438,premium
1,1,72.242914,0.053145,premium
2,2,31.601897,0.081376,basic
3,3,21.910621,0.088490,basic
4,4,4.070997,0.075453,basic


In [7]:
df['arm'] = np.random.choice(['control', 'treatment'], N, p=[0.502,0.498])

In [8]:
df.head(20)

,user_id,tenure_months,prior_engagement,account_type,arm
0,0,11.262434,0.050438,premium,treatment
1,1,72.242914,0.053145,premium,control
2,2,31.601897,0.081376,basic,treatment
3,3,21.910621,0.088490,basic,treatment
4,4,4.070997,0.075453,basic,treatment
5,5,4.070311,0.043916,basic,treatment
6,6,1.436130,0.023327,premium,control
7,7,48.269541,0.053019,basic,treatment
8,8,22.057972,0.072430,basic,control
9,9,29.550001,0.098126,basic,control


In [9]:
def sigmoid(x): return 1/(1+np.exp(-x))

In [10]:
base_logit = (
    -2.45
    + 1.5 * (df['prior_engagement'] - df['prior_engagement'].mean())
    + 0.05 * (df['tenure_months'] / 24)
)

In [11]:
true_effect_logit = 0.22

In [12]:
treatment = (df['arm'] == 'treatment').astype(int)

In [13]:
df['clicked'] = np.random.binomial(1, sigmoid(base_logit + true_effect_logit * treatment))

In [14]:
df['time-to-click'] = np.where(df['clicked'], np.random.exponential(45,N), np.nan)

In [15]:
df['confirmed_fraud'] = df['clicked'] * np.random.binomial(1,0.35,N)

In [16]:
df['unsubscribed'] = np.random.binomial(1,0.0025 + 0.0002 * treatment, N)

In [17]:
df['support ticket'] = np.random.binomial(1,0.012, N)

In [18]:
df['confirmed_fraud'].value_counts()

confirmed_fraud
0    77369
1     2631
Name: count, dtype: int64

In [19]:
df['unsubscribed'].value_counts()

unsubscribed
0    79791
1      209
Name: count, dtype: int64

In [20]:
df.head()

,user_id,tenure_months,prior_engagement,account_type,arm,clicked,time-to-click,confirmed_fraud,unsubscribed,support ticket
0,0,11.262434,0.050438,premium,treatment,0,NaN,0,0,0
1,1,72.242914,0.053145,premium,control,0,NaN,0,0,0
2,2,31.601897,0.081376,basic,treatment,0,NaN,0,0,0
3,3,21.910621,0.088490,basic,treatment,0,NaN,0,0,0
4,4,4.070997,0.075453,basic,treatment,0,NaN,0,0,0


In [21]:
df['support ticket'].value_counts()

support ticket
0    79014
1      986
Name: count, dtype: int64

In [22]:
df.groupby('arm')['clicked'].mean()

arm
control      0.084721
treatment    0.101775
Name: clicked, dtype: float64

In [23]:
df.groupby('arm').agg(
    n=('clicked', 'count'),
    clicks=('clicked', 'sum'),
    ctr=('clicked', 'mean')
)

,n,clicks,ctr
arm,,,
control,40285,3413,0.084721
treatment,39715,4042,0.101775


In [24]:
from scipy.stats import chisquare

In [25]:
counts = df['arm'].value_counts()
chi2, p = chisquare(counts)
print(counts, '\np-value', p)

arm
control      40285
treatment    39715
Name: count, dtype: int64 
p-value 0.04387800789041376


In [26]:
n_c = (df['arm'] == 'control').sum()
n_t = (df['arm'] == 'treatment').sum()

clicks_c = df.loc[df['arm'] == 'control', 'clicked'].sum()
clicks_t = df.loc[df['arm'] == 'treatment', 'clicked'].sum()

p_c = clicks_c / n_c
p_t = clicks_t / n_t

print(n_c, n_t, clicks_c, clicks_t, p_c, p_t)

40285 39715 3413 4042 0.08472136030780687 0.10177514792899409


In [27]:
diff = p_t - p_c

In [28]:
diff

np.float64(0.017053787621187214)

In [29]:
p_pool = (clicks_c + clicks_t) / (n_c + n_t)

In [30]:
import numpy as np

In [31]:
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t))

In [32]:
z = diff / se
print(z)

8.296353424695312


In [33]:
from scipy.stats import norm

In [34]:
p_value = 2 * (1-norm.cdf(abs(z)))

In [35]:
p_value

np.float64(0.0)

In [36]:
se_ci = np.sqrt(p_c * (1-p_c) / n_c + p_t * (1-p_t) / n_t)

In [37]:
ci_low = diff - 1.96 * se_ci
ci_high = diff + 1.96 * se_ci
print(ci_low, ci_high)

0.013024234865670791 0.021083340376703636


In [38]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

In [39]:
count = [clicks_t, clicks_c]
nobs = [n_t, n_c]

z_stat, p_val = proportions_ztest(count, nobs, alternative='two-sided')
print(f"z = {z_stat:.3f}, p = {p_val:2e}")

ci_low, ci_high = confint_proportions_2indep(clicks_t, n_t, clicks_c, n_c, method='wald')

print(f"95% CI for lift: [{ci_low:.4f}, {ci_high:.4f}]")

z = 8.296, p = 1.073558e-16
95% CI for lift: [0.0130, 0.0211]
